# Data on Australia

Requested by Alex at the end of 2024 due to the CBCR Law in Australia.

## 0. Load Packages

In [1]:
# Packages
import pandas as pd
import numpy as np
import tjn_tools
from config import *

# Show columns and select data format
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)
# Reset display.max_rows to default
pd.reset_option('display.max_rows')

pd.options.display.float_format = '{:,.2f}'.format

[TJN TOOLS: Data processing] Module loaded.
[TJN TOOLS: Other functions] Module loaded.
[TJN TOOLS: Paths] Module loaded. Sharepoint FOUND at /Users/mariocuendagarcia/Library/CloudStorage/OneDrive-SharedLibraries-TaxJusticeNetworkLtd


## 1. Define the Misalignment Formula

In [2]:
def calculate_misalignment(cbcr_data,
                           formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",
                                         'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip'],
                           weights=[.5, 0, 0, .5, 0, 0, 0, 0],
                           profit_var='profit_loss_before_income_tax_corrected',
                           etr_max=0.15): 

    # Create variable with positive profits only for calculating shares
    cbcr_data['profit_var_pos'] = cbcr_data[profit_var]
    cbcr_data.loc[cbcr_data[profit_var] < 0, 'profit_var_pos'] = 0
    cbcr_data['share_profit'] = cbcr_data['profit_var_pos'] / cbcr_data.groupby('iso_parent')['profit_var_pos'].transform('sum')

    # Calculate weighted shares of economic activity
    actual_weights = []
    actual_variables = []
    for i, var in enumerate(formula_vars):
        if var is not None and weights[i] > 0:
            actual_variables.append(f"share_{var}")
            actual_weights.append(weights[i])
            cbcr_data.loc[cbcr_data[var] < 0, var] = 0  # Set economic activity measure to zero if negative
            cbcr_data[f"share_{var}"] = cbcr_data[var] / cbcr_data.groupby('iso_parent')[var].transform('sum')

    # Calculate the share of economic activity
    cbcr_data["share_economy_partner_of_parent"] = (cbcr_data.loc[:, actual_variables] * actual_weights).sum(1, min_count=len(actual_weights))
    # Set economic activity to 1% for those jurisdictions without economic activity but with reported profits
    cbcr_data.loc[(cbcr_data["share_economy_partner_of_parent"] == 0) & (cbcr_data[profit_var] > 0), "share_economy_partner_of_parent"] = 0.01

    # Normalize the economic activity shares to sum to 1
    cbcr_data["share_economy_partner_of_parent"] = cbcr_data["share_economy_partner_of_parent"] / cbcr_data.groupby('iso_parent')["share_economy_partner_of_parent"].transform('sum')

    # Calculate theoretical profit and misaligned profit
    cbcr_data["theoretical_profit"] = cbcr_data["share_economy_partner_of_parent"] * cbcr_data.groupby('iso_parent')[profit_var].transform('sum')
    cbcr_data["misaligned_profit"] = cbcr_data[profit_var] - cbcr_data["theoretical_profit"]

    # Set positive misaligned profits to 0 if ETR exceeds the threshold (etr_max)
    cbcr_data.loc[((cbcr_data["misaligned_profit"] > 0) & (cbcr_data["etr_average_corrected"] > etr_max)), "misaligned_profit"] = 0

    # Adjust misalignment per 'iso_parent'
    def adjust_misalignment(group):
        total_negative_misalignment = group.loc[group["misaligned_profit"] < 0, "misaligned_profit"].sum()
        total_positive_misalignment = group.loc[group["misaligned_profit"] > 0, "misaligned_profit"].sum()
        
        # Adjust negative misalignments to balance positive misalignments within each 'iso_parent'
        if total_negative_misalignment != 0:
            factor = - total_positive_misalignment / total_negative_misalignment
            group.loc[group["misaligned_profit"] < 0, "misaligned_profit"] *= factor
        
        return group

    # Apply the adjustment by grouping by 'iso_parent'
    cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)

    return cbcr_data

## Australia in 2016

In [13]:
# Load the data
final_misalignment_2016 = pd.read_csv(f'{output_tables}/Datasets_to_Perform_Analysis/misalignment_dataset_2016.csv') 

# Start the estimates
misalignment_final_estimates_australia_2016 = final_misalignment_2016[final_misalignment_2016['year'] == 2016].copy()
misalignment_final_estimates_australia_2016 = calculate_misalignment(misalignment_final_estimates_australia_2016, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Generate a new column called negative_misalignment equal to misaligned_profit if misaligned_profit < 0 and a new column called positive_misalignment equal to misaligned_profit if misaligned_profit > 0
misalignment_final_estimates_australia_2016['negative_misalignment'] = misalignment_final_estimates_australia_2016['misaligned_profit'].apply(lambda x: x if x < 0 else 0)
misalignment_final_estimates_australia_2016['positive_misalignment'] = misalignment_final_estimates_australia_2016['misaligned_profit'].apply(lambda x: x if x > 0 else 0)

# Rename profit_loss_before_income_tax_corrected to reported_profit
misalignment_final_estimates_australia_2016 = misalignment_final_estimates_australia_2016.rename(columns={'profit_loss_before_income_tax_corrected': 'reported_profit'})

# Convert results to millions
misalignment_final_estimates_australia_2016['negative_misalignment'] = -misalignment_final_estimates_australia_2016['negative_misalignment'] / 1e6
misalignment_final_estimates_australia_2016['positive_misalignment'] = misalignment_final_estimates_australia_2016['positive_misalignment'] / 1e6
misalignment_final_estimates_australia_2016['theoretical_profit'] = misalignment_final_estimates_australia_2016['theoretical_profit'] / 1e6
misalignment_final_estimates_australia_2016['reported_profit'] = misalignment_final_estimates_australia_2016['reported_profit'] / 1e6

# Calculate other relevant variables
misalignment_final_estimates_australia_2016['tax_revenue_loss'] = misalignment_final_estimates_australia_2016['negative_misalignment'] * misalignment_final_estimates_australia_2016['cit']
misalignment_final_estimates_australia_2016['tax_revenue_gain'] = misalignment_final_estimates_australia_2016['positive_misalignment'] * misalignment_final_estimates_australia_2016['etr_average_corrected']

# Keep iso_partner, iso_parent, year, negative_misalignment, positive_misalignment, tax_revenue_loss, tax_revenue gain
misalignment_final_estimates_australia_2016 = misalignment_final_estimates_australia_2016[['iso_parent', 'iso_partner', 'year', 'cit', 'theoretical_profit', 'reported_profit', 'negative_misalignment', 'positive_misalignment', 'tax_revenue_loss', 'tax_revenue_gain']]

# Sort by iso_partner and then iso_parent
misalignment_final_estimates_australia_2016 = misalignment_final_estimates_australia_2016.sort_values(by=['iso_parent', 'iso_partner'])   

# Display 8 decimals
pd.options.display.float_format = '{:,.8f}'.format
pd.set_option('display.max_rows', None)
misalignment_final_estimates_australia_2016[(misalignment_final_estimates_australia_2016['iso_parent'] == 'AUS')] #| (misalignment_final_estimates_australia_2016['iso_partner'] == 'AUS')]


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_7742/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,year,cit,theoretical_profit,reported_profit,negative_misalignment,positive_misalignment,tax_revenue_loss,tax_revenue_gain
0,AUS,ARE,2016,0.00000000,51.47280536,9.66172799,37.51863043,0.00000000,0.00000000,0.00000000
1,AUS,ARG,2016,0.35000000,41.56116090,72.58364126,-0.00000000,0.00000000,-0.00000000,0.00000000
2,AUS,AUS,2016,0.30000000,"45,783.25704388","43,361.75250940","2,172.90582893",0.00000000,651.87174868,0.00000000
3,AUS,AUT,2016,0.25000000,6.22629002,5.86374104,0.32532865,0.00000000,0.08133216,0.00000000
4,AUS,BEL,2016,0.33990000,59.80166833,70.33835017,-0.00000000,0.00000000,-0.00000000,0.00000000
5,AUS,BMU,2016,0.00000000,1.62304933,159.35905649,-0.00000000,157.73600716,-0.00000000,1.76610375
6,AUS,BRA,2016,0.34000000,80.39614786,-246.60125210,293.42689485,0.00000000,99.76514425,0.00000000
7,AUS,CAN,2016,0.26700000,402.06291084,31.79837777,332.25209811,0.00000000,88.71131019,0.00000000
8,AUS,CHE,2016,0.21148581,243.48242433,"1,135.91157953",-0.00000000,892.42915520,-0.00000000,78.47174200
9,AUS,CHL,2016,0.24000000,561.10419902,727.28969490,-0.00000000,0.00000000,-0.00000000,0.00000000


In [14]:
# Count how many values in misalignment_final_estimates_australia_2016['iso_parent'] == 'AUS'
misalignment_australia_2016 = misalignment_final_estimates_australia_2016[misalignment_final_estimates_australia_2016['iso_parent'] == 'AUS']

# Count total values in misalignment_australia
misalignment_australia_2016['iso_partner'].count()

# Keep if positive_misalignment > 0
misalignment_australia_2016 = misalignment_australia_2016[misalignment_australia_2016['positive_misalignment'] > 0]
misalignment_australia_2016

# Export the results to excel
misalignment_australia_2016.to_csv(f'{output_tables}/Australia/australia_request_2016.csv', index=False)

## Australia in 2017

In [11]:
# Load the data
final_misalignment_2017 = pd.read_csv(f'{output_tables}/Datasets_to_Perform_Analysis/misalignment_dataset_2017.csv') 

# Start the estimates
misalignment_final_estimates_australia_2017 = final_misalignment_2017[final_misalignment_2017['year'] == 2017].copy()
misalignment_final_estimates_australia_2017 = calculate_misalignment(misalignment_final_estimates_australia_2017, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Generate a new column called negative_misalignment equal to misaligned_profit if misaligned_profit < 0 and a new column called positive_misalignment equal to misaligned_profit if misaligned_profit > 0
misalignment_final_estimates_australia_2017['negative_misalignment'] = misalignment_final_estimates_australia_2017['misaligned_profit'].apply(lambda x: x if x < 0 else 0)
misalignment_final_estimates_australia_2017['positive_misalignment'] = misalignment_final_estimates_australia_2017['misaligned_profit'].apply(lambda x: x if x > 0 else 0)

# Rename profit_loss_before_income_tax_corrected to reported_profit
misalignment_final_estimates_australia_2017 = misalignment_final_estimates_australia_2017.rename(columns={'profit_loss_before_income_tax_corrected': 'reported_profit'})

# Convert results to millions
misalignment_final_estimates_australia_2017['negative_misalignment'] = -misalignment_final_estimates_australia_2017['negative_misalignment'] / 1e6
misalignment_final_estimates_australia_2017['positive_misalignment'] = misalignment_final_estimates_australia_2017['positive_misalignment'] / 1e6
misalignment_final_estimates_australia_2017['theoretical_profit'] = misalignment_final_estimates_australia_2017['theoretical_profit'] / 1e6
misalignment_final_estimates_australia_2017['reported_profit'] = misalignment_final_estimates_australia_2017['reported_profit'] / 1e6

# Calculate other relevant variables
misalignment_final_estimates_australia_2017['tax_revenue_loss'] = misalignment_final_estimates_australia_2017['negative_misalignment'] * misalignment_final_estimates_australia_2017['cit']
misalignment_final_estimates_australia_2017['tax_revenue_gain'] = misalignment_final_estimates_australia_2017['positive_misalignment'] * misalignment_final_estimates_australia_2017['etr_average_corrected']

# Keep iso_partner, iso_parent, year, negative_misalignment, positive_misalignment, tax_revenue_loss, tax_revenue gain
misalignment_final_estimates_australia_2017 = misalignment_final_estimates_australia_2017[['iso_parent', 'iso_partner', 'year', 'cit', 'theoretical_profit', 'reported_profit', 'negative_misalignment', 'positive_misalignment', 'tax_revenue_loss', 'tax_revenue_gain']]

# Sort by iso_partner and then iso_parent
misalignment_final_estimates_australia_2017 = misalignment_final_estimates_australia_2017.sort_values(by=['iso_parent', 'iso_partner'])   

# Display 8 decimals
pd.options.display.float_format = '{:,.8f}'.format
pd.set_option('display.max_rows', None)
misalignment_final_estimates_australia_2017[(misalignment_final_estimates_australia_2017['iso_parent'] == 'AUS')] #| (misalignment_final_estimates_australia_2017['iso_partner'] == 'AUS')]


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_7742/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,year,cit,theoretical_profit,reported_profit,negative_misalignment,positive_misalignment,tax_revenue_loss,tax_revenue_gain
15,AUS,ARE,2017,0.00000000,64.75702417,9.50827241,40.25891238,0.00000000,0.00000000,0.00000000
16,AUS,ARG,2017,0.35000000,56.13390061,55.62791700,0.36870244,0.00000000,0.12904586,0.00000000
17,AUS,AUS,2017,0.30000000,"58,603.32513598","57,758.07272235",615.92238337,0.00000000,184.77671501,0.00000000
18,AUS,AUT,2017,0.25000000,16.34780060,15.15843000,0.86667599,0.00000000,0.21666900,0.00000000
19,AUS,BEL,2017,0.33990000,76.94101145,93.35817200,-0.00000000,0.00000000,-0.00000000,0.00000000
20,AUS,BGR,2017,0.10000000,4.42091124,2.51819400,1.38648068,0.00000000,0.13864807,0.00000000
21,AUS,BMU,2017,0.00000000,1.83588500,-478.23357800,349.81920556,0.00000000,0.00000000,0.00000000
22,AUS,BRA,2017,0.34000000,127.59448450,-223.35896600,255.73436076,0.00000000,86.94968266,0.00000000
23,AUS,CAN,2017,0.26650000,559.67508232,137.72225100,307.47051336,0.00000000,81.94089181,0.00000000
24,AUS,CHE,2017,0.21148581,264.05260632,"1,142.59185222",-0.00000000,878.53924590,-0.00000000,70.66608709


In [12]:
# Count how many values in misalignment_final_estimates_australia_2017['iso_parent'] == 'AUS'
misalignment_australia_2017 = misalignment_final_estimates_australia_2017[misalignment_final_estimates_australia_2017['iso_parent'] == 'AUS']

# Count total values in misalignment_australia
misalignment_australia_2017['iso_partner'].count()

# Keep if positive_misalignment > 0
misalignment_australia_2017 = misalignment_australia_2017[misalignment_australia_2017['positive_misalignment'] > 0]
misalignment_australia_2017

# Export the results to excel
misalignment_australia_2017.to_csv(f'{output_tables}/Australia/australia_request_2017.csv', index=False)

## Australia in 2018

In [9]:
# Load the data
final_misalignment_2018 = pd.read_csv(f'{output_tables}/Datasets_to_Perform_Analysis/misalignment_dataset_2018.csv') 

# Start the estimates
misalignment_final_estimates_australia_2018 = final_misalignment_2018[final_misalignment_2018['year'] == 2018].copy()
misalignment_final_estimates_australia_2018 = calculate_misalignment(misalignment_final_estimates_australia_2018, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Generate a new column called negative_misalignment equal to misaligned_profit if misaligned_profit < 0 and a new column called positive_misalignment equal to misaligned_profit if misaligned_profit > 0
misalignment_final_estimates_australia_2018['negative_misalignment'] = misalignment_final_estimates_australia_2018['misaligned_profit'].apply(lambda x: x if x < 0 else 0)
misalignment_final_estimates_australia_2018['positive_misalignment'] = misalignment_final_estimates_australia_2018['misaligned_profit'].apply(lambda x: x if x > 0 else 0)

# Rename profit_loss_before_income_tax_corrected to reported_profit
misalignment_final_estimates_australia_2018 = misalignment_final_estimates_australia_2018.rename(columns={'profit_loss_before_income_tax_corrected': 'reported_profit'})

# Convert results to millions
misalignment_final_estimates_australia_2018['negative_misalignment'] = -misalignment_final_estimates_australia_2018['negative_misalignment'] / 1e6
misalignment_final_estimates_australia_2018['positive_misalignment'] = misalignment_final_estimates_australia_2018['positive_misalignment'] / 1e6
misalignment_final_estimates_australia_2018['theoretical_profit'] = misalignment_final_estimates_australia_2018['theoretical_profit'] / 1e6
misalignment_final_estimates_australia_2018['reported_profit'] = misalignment_final_estimates_australia_2018['reported_profit'] / 1e6

# Calculate other relevant variables
misalignment_final_estimates_australia_2018['tax_revenue_loss'] = misalignment_final_estimates_australia_2018['negative_misalignment'] * misalignment_final_estimates_australia_2018['cit']
misalignment_final_estimates_australia_2018['tax_revenue_gain'] = misalignment_final_estimates_australia_2018['positive_misalignment'] * misalignment_final_estimates_australia_2018['etr_average_corrected']

# Keep iso_partner, iso_parent, year, negative_misalignment, positive_misalignment, tax_revenue_loss, tax_revenue gain
misalignment_final_estimates_australia_2018 = misalignment_final_estimates_australia_2018[['iso_parent', 'iso_partner', 'year', 'cit', 'theoretical_profit', 'reported_profit', 'negative_misalignment', 'positive_misalignment', 'tax_revenue_loss', 'tax_revenue_gain']]

# Sort by iso_partner and then iso_parent
misalignment_final_estimates_australia_2018 = misalignment_final_estimates_australia_2018.sort_values(by=['iso_parent', 'iso_partner'])   

# Display 8 decimals
pd.options.display.float_format = '{:,.8f}'.format
pd.set_option('display.max_rows', None)
misalignment_final_estimates_australia_2018[(misalignment_final_estimates_australia_2018['iso_parent'] == 'AUS')] #| (misalignment_final_estimates_australia_2018['iso_partner'] == 'AUS')]


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_7742/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,year,cit,theoretical_profit,reported_profit,negative_misalignment,positive_misalignment,tax_revenue_loss,tax_revenue_gain
19,AUS,ARE,2018,0.00000000,20.81821725,9.27104997,6.18094742,0.00000000,0.00000000,0.00000000
20,AUS,ARG,2018,0.30000000,39.11772964,25.64693800,7.21062169,0.00000000,2.16318651,0.00000000
21,AUS,AUS,2018,0.30000000,"69,234.72606312","58,131.80672975","5,943.15115727",0.00000000,"1,782.94534718",0.00000000
22,AUS,AUT,2018,0.25000000,17.28733549,19.03805500,-0.00000000,1.75071951,-0.00000000,0.17363673
23,AUS,BEL,2018,0.29580000,84.34460437,135.85460300,-0.00000000,0.00000000,-0.00000000,0.00000000
24,AUS,BGR,2018,0.10000000,3.95739059,3.29226900,0.35602512,0.00000000,0.03560251,0.00000000
25,AUS,BMU,2018,0.00000000,2.09081366,332.33614050,-0.00000000,330.24532684,-0.00000000,4.40443229
26,AUS,BRA,2018,0.34000000,161.79142816,-437.41080900,320.73992094,0.00000000,109.05157312,0.00000000
27,AUS,BWA,2018,0.22000000,0.03070341,2.01189200,-0.00000000,0.00000000,-0.00000000,0.00000000
28,AUS,CAN,2018,0.26780000,658.65267238,465.37668000,103.45643369,0.00000000,27.70563294,0.00000000


In [ ]:
# Count how many values in misalignment_final_estimates_australia_2018['iso_parent'] == 'AUS'
misalignment_australia_2018 = misalignment_final_estimates_australia_2018[misalignment_final_estimates_australia_2018['iso_parent'] == 'AUS']

# Count total values in misalignment_australia
misalignment_australia_2018['iso_partner'].count()

# Keep if positive_misalignment > 0
misalignment_australia_2018 = misalignment_australia_2018[misalignment_australia_2018['positive_misalignment'] > 0]
misalignment_australia_2018

# Export the results to excel
misalignment_australia_2018.to_csv(f'{output_tables}/Australia/australia_request_2018.csv', index=False)

## Australia in 2019

In [7]:
# Load the data
final_misalignment_2019 = pd.read_csv(f'{output_tables}/Datasets_to_Perform_Analysis/misalignment_dataset_2019.csv') 

# Start the estimates
misalignment_final_estimates_australia_2019 = final_misalignment_2019[final_misalignment_2019['year'] == 2019].copy()
misalignment_final_estimates_australia_2019 = calculate_misalignment(misalignment_final_estimates_australia_2019, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Generate a new column called negative_misalignment equal to misaligned_profit if misaligned_profit < 0 and a new column called positive_misalignment equal to misaligned_profit if misaligned_profit > 0
misalignment_final_estimates_australia_2019['negative_misalignment'] = misalignment_final_estimates_australia_2019['misaligned_profit'].apply(lambda x: x if x < 0 else 0)
misalignment_final_estimates_australia_2019['positive_misalignment'] = misalignment_final_estimates_australia_2019['misaligned_profit'].apply(lambda x: x if x > 0 else 0)

# Rename profit_loss_before_income_tax_corrected to reported_profit
misalignment_final_estimates_australia_2019 = misalignment_final_estimates_australia_2019.rename(columns={'profit_loss_before_income_tax_corrected': 'reported_profit'})

# Convert results to millions
misalignment_final_estimates_australia_2019['negative_misalignment'] = -misalignment_final_estimates_australia_2019['negative_misalignment'] / 1e6
misalignment_final_estimates_australia_2019['positive_misalignment'] = misalignment_final_estimates_australia_2019['positive_misalignment'] / 1e6
misalignment_final_estimates_australia_2019['theoretical_profit'] = misalignment_final_estimates_australia_2019['theoretical_profit'] / 1e6
misalignment_final_estimates_australia_2019['reported_profit'] = misalignment_final_estimates_australia_2019['reported_profit'] / 1e6

# Calculate other relevant variables
misalignment_final_estimates_australia_2019['tax_revenue_loss'] = misalignment_final_estimates_australia_2019['negative_misalignment'] * misalignment_final_estimates_australia_2019['cit']
misalignment_final_estimates_australia_2019['tax_revenue_gain'] = misalignment_final_estimates_australia_2019['positive_misalignment'] * misalignment_final_estimates_australia_2019['etr_average_corrected']

# Keep iso_partner, iso_parent, year, negative_misalignment, positive_misalignment, tax_revenue_loss, tax_revenue gain
misalignment_final_estimates_australia_2019 = misalignment_final_estimates_australia_2019[['iso_parent', 'iso_partner', 'year', 'cit', 'theoretical_profit', 'reported_profit', 'negative_misalignment', 'positive_misalignment', 'tax_revenue_loss', 'tax_revenue_gain']]

# Sort by iso_partner and then iso_parent
misalignment_final_estimates_australia_2019 = misalignment_final_estimates_australia_2019.sort_values(by=['iso_parent', 'iso_partner'])   

# Display 8 decimals
pd.options.display.float_format = '{:,.8f}'.format
pd.set_option('display.max_rows', None)
misalignment_final_estimates_australia_2019[(misalignment_final_estimates_australia_2019['iso_parent'] == 'AUS')] #| (misalignment_final_estimates_australia_2019['iso_partner'] == 'AUS')]


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_7742/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,year,cit,theoretical_profit,reported_profit,negative_misalignment,positive_misalignment,tax_revenue_loss,tax_revenue_gain
18,AUS,ARE,2019,0.00000000,56.40537971,23.10542900,22.27619285,0.00000000,0.00000000,0.00000000
19,AUS,ARG,2019,0.30000000,27.43558313,6.11199000,14.26453982,0.00000000,4.27936195,0.00000000
20,AUS,AUS,2019,0.30000000,"53,968.78810774","53,840.68670500",85.69416745,0.00000000,25.70825024,0.00000000
21,AUS,AUT,2019,0.25000000,17.22347935,24.17682300,-0.00000000,6.95334365,-0.00000000,0.67001975
22,AUS,BEL,2019,0.29580000,125.11986445,66.98999900,38.88630660,0.00000000,11.50256949,0.00000000
23,AUS,BGR,2019,0.10000000,5.31355236,-24.55420400,19.98020677,0.00000000,1.99802068,0.00000000
24,AUS,BMU,2019,0.00000000,1.85393265,308.98186100,-0.00000000,307.12792835,-0.00000000,3.83247428
25,AUS,BRA,2019,0.34000000,121.24026773,38.34814300,55.45116185,0.00000000,18.85339503,0.00000000
26,AUS,BWA,2019,0.22000000,0.67866017,2.38632800,-0.00000000,0.00000000,-0.00000000,0.00000000
27,AUS,CAN,2019,0.26620000,866.63521880,-528.79439200,933.48063466,0.00000000,248.49254495,0.00000000


In [8]:
# Count how many values in misalignment_final_estimates_australia_2019['iso_parent'] == 'AUS'
misalignment_australia_2019 = misalignment_final_estimates_australia_2019[misalignment_final_estimates_australia_2019['iso_parent'] == 'AUS']

# Count total values in misalignment_australia
misalignment_australia_2019['iso_partner'].count()

# Keep if positive_misalignment > 0
misalignment_australia_2019 = misalignment_australia_2019[misalignment_australia_2019['positive_misalignment'] > 0]
misalignment_australia_2019

# Export the results to excel
misalignment_australia_2019.to_csv(f'{output_tables}/Australia/australia_request_2019.csv', index=False)

## Australia in 2020

In [5]:
# Load the data
final_misalignment_2020 = pd.read_csv(f'{output_tables}/Datasets_to_Perform_Analysis/misalignment_dataset_2020.csv') 

# Start the estimates
misalignment_final_estimates_australia_2020 = final_misalignment_2020[final_misalignment_2020['year'] == 2020].copy()
misalignment_final_estimates_australia_2020 = calculate_misalignment(misalignment_final_estimates_australia_2020, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Generate a new column called negative_misalignment equal to misaligned_profit if misaligned_profit < 0 and a new column called positive_misalignment equal to misaligned_profit if misaligned_profit > 0
misalignment_final_estimates_australia_2020['negative_misalignment'] = misalignment_final_estimates_australia_2020['misaligned_profit'].apply(lambda x: x if x < 0 else 0)
misalignment_final_estimates_australia_2020['positive_misalignment'] = misalignment_final_estimates_australia_2020['misaligned_profit'].apply(lambda x: x if x > 0 else 0)

# Rename profit_loss_before_income_tax_corrected to reported_profit
misalignment_final_estimates_australia_2020 = misalignment_final_estimates_australia_2020.rename(columns={'profit_loss_before_income_tax_corrected': 'reported_profit'})

# Convert results to millions
misalignment_final_estimates_australia_2020['negative_misalignment'] = -misalignment_final_estimates_australia_2020['negative_misalignment'] / 1e6
misalignment_final_estimates_australia_2020['positive_misalignment'] = misalignment_final_estimates_australia_2020['positive_misalignment'] / 1e6
misalignment_final_estimates_australia_2020['theoretical_profit'] = misalignment_final_estimates_australia_2020['theoretical_profit'] / 1e6
misalignment_final_estimates_australia_2020['reported_profit'] = misalignment_final_estimates_australia_2020['reported_profit'] / 1e6

# Calculate other relevant variables
misalignment_final_estimates_australia_2020['tax_revenue_loss'] = misalignment_final_estimates_australia_2020['negative_misalignment'] * misalignment_final_estimates_australia_2020['cit']
misalignment_final_estimates_australia_2020['tax_revenue_gain'] = misalignment_final_estimates_australia_2020['positive_misalignment'] * misalignment_final_estimates_australia_2020['etr_average_corrected']

# Keep iso_partner, iso_parent, year, negative_misalignment, positive_misalignment, tax_revenue_loss, tax_revenue gain
misalignment_final_estimates_australia_2020 = misalignment_final_estimates_australia_2020[['iso_parent', 'iso_partner', 'year', 'cit', 'theoretical_profit', 'reported_profit', 'negative_misalignment', 'positive_misalignment', 'tax_revenue_loss', 'tax_revenue_gain']]

# Sort by iso_partner and then iso_parent
misalignment_final_estimates_australia_2020 = misalignment_final_estimates_australia_2020.sort_values(by=['iso_parent', 'iso_partner'])   

# Display 8 decimals
pd.options.display.float_format = '{:,.8f}'.format
pd.set_option('display.max_rows', None)
misalignment_final_estimates_australia_2020[(misalignment_final_estimates_australia_2020['iso_parent'] == 'AUS')] #| (misalignment_final_estimates_australia_2020['iso_partner'] == 'AUS')]


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_7742/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,year,cit,theoretical_profit,reported_profit,negative_misalignment,positive_misalignment,tax_revenue_loss,tax_revenue_gain
17,AUS,ARE,2020,0.00000000,48.83834046,4.45462587,18.80006457,0.00000000,0.00000000,0.00000000
18,AUS,ARG,2020,0.30000000,38.15287173,52.32911780,-0.00000000,0.00000000,-0.00000000,0.00000000
19,AUS,AUS,2020,0.30000000,"67,318.17155651","73,328.77534400",-0.00000000,0.00000000,-0.00000000,0.00000000
20,AUS,AUT,2020,0.25000000,29.18371749,-26.38515424,23.53787614,0.00000000,5.88446904,0.00000000
21,AUS,BEL,2020,0.25000000,110.81808247,96.81298491,5.93228262,0.00000000,1.48307065,0.00000000
22,AUS,BGR,2020,0.10000000,7.07102467,28.09674011,-0.00000000,21.02571544,-0.00000000,1.64705981
23,AUS,BMU,2020,0.00000000,1.90300661,183.79405590,-0.00000000,181.89104929,-0.00000000,2.31614696
24,AUS,BRA,2020,0.34000000,128.42084651,-149.57894820,117.75522034,0.00000000,40.03677491,0.00000000
25,AUS,BWA,2020,0.22000000,5.93600835,0.31734573,2.37995447,0.00000000,0.52358998,0.00000000
26,AUS,CAN,2020,0.26250000,852.60595861,-787.87156200,694.87386528,0.00000000,182.40438964,0.00000000


In [6]:
# Count how many values in misalignment_final_estimates_australia_2020['iso_parent'] == 'AUS'
misalignment_australia_2020 = misalignment_final_estimates_australia_2020[misalignment_final_estimates_australia_2020['iso_parent'] == 'AUS']

# Count total values in misalignment_australia
misalignment_australia_2020['iso_partner'].count()

# Keep if positive_misalignment > 0
misalignment_australia_2020 = misalignment_australia_2020[misalignment_australia_2020['positive_misalignment'] > 0]
misalignment_australia_2020

# Export the results to excel
misalignment_australia_2020.to_csv(f'{output_tables}/Australia/australia_request_2020.csv', index=False)

## Australia in 2021

In [3]:
# Load the data
final_misalignment_2021 = pd.read_csv(f'{output_tables}/Datasets_to_Perform_Analysis/misalignment_dataset_2021.csv') 

# Start the estimates
misalignment_final_estimates_australia_2021 = final_misalignment_2021[final_misalignment_2021['year'] == 2021].copy()
misalignment_final_estimates_australia_2021 = calculate_misalignment(misalignment_final_estimates_australia_2021, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Generate a new column called negative_misalignment equal to misaligned_profit if misaligned_profit < 0 and a new column called positive_misalignment equal to misaligned_profit if misaligned_profit > 0
misalignment_final_estimates_australia_2021['negative_misalignment'] = misalignment_final_estimates_australia_2021['misaligned_profit'].apply(lambda x: x if x < 0 else 0)
misalignment_final_estimates_australia_2021['positive_misalignment'] = misalignment_final_estimates_australia_2021['misaligned_profit'].apply(lambda x: x if x > 0 else 0)

# Rename profit_loss_before_income_tax_corrected to reported_profit
misalignment_final_estimates_australia_2021 = misalignment_final_estimates_australia_2021.rename(columns={'profit_loss_before_income_tax_corrected': 'reported_profit'})

# Convert results to millions
misalignment_final_estimates_australia_2021['negative_misalignment'] = -misalignment_final_estimates_australia_2021['negative_misalignment'] / 1e6
misalignment_final_estimates_australia_2021['positive_misalignment'] = misalignment_final_estimates_australia_2021['positive_misalignment'] / 1e6
misalignment_final_estimates_australia_2021['theoretical_profit'] = misalignment_final_estimates_australia_2021['theoretical_profit'] / 1e6
misalignment_final_estimates_australia_2021['reported_profit'] = misalignment_final_estimates_australia_2021['reported_profit'] / 1e6

# Calculate other relevant variables
misalignment_final_estimates_australia_2021['tax_revenue_loss'] = misalignment_final_estimates_australia_2021['negative_misalignment'] * misalignment_final_estimates_australia_2021['cit']
misalignment_final_estimates_australia_2021['tax_revenue_gain'] = misalignment_final_estimates_australia_2021['positive_misalignment'] * misalignment_final_estimates_australia_2021['etr_average_corrected']

# Keep iso_partner, iso_parent, year, negative_misalignment, positive_misalignment, tax_revenue_loss, tax_revenue gain
misalignment_final_estimates_australia_2021 = misalignment_final_estimates_australia_2021[['iso_parent', 'iso_partner', 'year', 'cit', 'theoretical_profit', 'reported_profit', 'negative_misalignment', 'positive_misalignment', 'tax_revenue_loss', 'tax_revenue_gain']]

# Sort by iso_partner and then iso_parent
misalignment_final_estimates_australia_2021 = misalignment_final_estimates_australia_2021.sort_values(by=['iso_parent', 'iso_partner'])   

# Display 8 decimals
pd.options.display.float_format = '{:,.8f}'.format
pd.set_option('display.max_rows', None)
misalignment_final_estimates_australia_2021[(misalignment_final_estimates_australia_2021['iso_parent'] == 'AUS')] #| (misalignment_final_estimates_australia_2021['iso_partner'] == 'AUS')]


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_7742/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,year,cit,theoretical_profit,reported_profit,negative_misalignment,positive_misalignment,tax_revenue_loss,tax_revenue_gain
168,AUS,ARE,2021,0.00000000,68.74847172,17.53242900,15.70545568,0.00000000,0.00000000,0.00000000
169,AUS,ARG,2021,0.30000000,45.09391400,17.31457100,8.51856600,0.00000000,2.55556980,0.00000000
170,AUS,AUS,2021,0.30000000,"89,671.85985969","95,005.12027700",-0.00000000,0.00000000,-0.00000000,0.00000000
171,AUS,AUT,2021,0.25000000,21.71374916,18.83311000,0.88335116,0.00000000,0.22083779,0.00000000
172,AUS,BEL,2021,0.25000000,157.71122982,207.31682900,-0.00000000,49.60559918,-0.00000000,6.86889198
173,AUS,BGR,2021,0.10000000,10.08242509,18.14296600,-0.00000000,8.06054091,-0.00000000,0.68007429
174,AUS,BMU,2021,0.00000000,2.05013955,441.42198600,-0.00000000,439.37184645,-0.00000000,6.22651597
175,AUS,BRA,2021,0.34000000,163.26747484,19.71960200,44.01911265,0.00000000,14.96649830,0.00000000
176,AUS,BWA,2021,0.22000000,13.90458954,23.98282900,-0.00000000,0.00000000,-0.00000000,0.00000000
177,AUS,CAN,2021,0.26170000,"1,309.28379413",-613.27364200,589.55434648,0.00000000,154.28637247,0.00000000


In [4]:
# Count how many values in misalignment_final_estimates_spain_2021['iso_parent'] == 'AUS'
misalignment_australia_2021 = misalignment_final_estimates_australia_2021[misalignment_final_estimates_australia_2021['iso_parent'] == 'AUS']

# Count total values in misalignment_australia
misalignment_australia_2021['iso_partner'].count()

# Keep if positive_misalignment > 0
misalignment_australia_2021 = misalignment_australia_2021[misalignment_australia_2021['positive_misalignment'] > 0]
misalignment_australia_2021

# Export the results to excel
misalignment_australia_2021.to_csv(f'{output_tables}/Australia/australia_request_2021.csv', index=False)